# Summertime Map

Every recording here is the same song, so the differences the map picks up come from
interpretation: tempo, energy, instrumentation, mood.

The code that fetches and prepares the data lives in the `summertime/` package; this
notebook runs each step and explains it. To bring the data up to date after adding
songs to the playlist, run `python -m summertime.update` from the project folder, then
re-run this notebook.

Steps:
1. Load the song list
2. Find each recording on Deezer or iTunes (album cover + 30-second preview)
3. Audio features from each preview
4. Standardize
5. PCA: the 2-D map
6. K-Means: are there natural groups?
7. How valid are the groups?
8. What might each group represent?
9. Export for the website

In [1]:
import sys
sys.path.insert(0, "..")  # make the summertime package importable from notebooks/

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from summertime.data import FEATURES, load_songs
from summertime.features import load_dataset, update_features
from summertime.matching import match_report, update_matches

## 1. Load the song list

`data/songs.csv` has one row per recording (see `summertime/data.py`). The Spotify
track ID is taken from each link.

In [2]:
songs = load_songs()
print(f"{len(songs)} recordings, {songs['year'].notna().sum()} with a year")
songs.drop(columns=["spotify_url"])

54 recordings, 28 with a year


,artist,title,year,year_source,duration_s,notes,track_id
0,Big Brother & The Holding Company;Janis Joplin,Summertime,1968,draft - verify,241,NaN,3eYCJXTViFwl5mdtcTsF4G
1,Ella Fitzgerald;Louis Armstrong,Summertime,1957,draft - verify,298,NaN,2gNjmvuQiEd2z9SqyYi8HH
2,The Zombies,Summertime,1965,draft - verify,138,NaN,7egTKPYxbWaaV1jEzb1rHu
3,Sam Cooke,Summertime,1957,draft - verify,143,NaN,5blqPmozIU2vhHN7vbBKxk
4,Sublime,Doin' Time,1996,draft - verify,251,based on Summertime,4mAqG6KNWZQiekSthr2QtL
5,Charlie Parker,Summertime,1949,draft - verify,167,NaN,6cU4urQRUieTBH08yKHAqj
6,Al Green,Summertime,1969,draft - verify,179,NaN,1wgyVw5XL14SW2BV4lBcVi
7,Lana Del Rey,Doin' Time,2019,draft - verify,202,cover of Sublime's Doin' Time,0cLS7FslBJgsuWwJNRnPWB
8,Doc Watson;David Grisman,Summertime,1997,draft - verify,260,NaN,5xV7bCTvuIcxvB1NcjshNY
9,Billy Stewart,Summertime,1966,draft - verify,255,NaN,4qH8fQ5Ub2r6llHnrRGM3w


## 2. Find each recording on Deezer (or iTunes)

Code: `summertime/matching.py`.

Spotify no longer gives new apps preview clips, but Deezer does, along with album covers.
Many songs exist in several versions (remasters, live takes, instrumentals), so for each
song we search Deezer and keep the candidate that:

1. is by the same artist and has the same title,
2. has the same version tags (*instrumental, remix, live, dub...*) as the Spotify title, and
3. has the length closest to the Spotify track's.

Matches are saved in `data/matches.csv` and reused on later runs, so you can fix a wrong one by
editing its `source` and `source_id` there (the number at the end of the track's deezer.com
or music.apple.com link).

**Check the table below by eye**, starting with the biggest length differences: a wrong
match means the wrong recording gets analyzed.

If Deezer doesn't have a recording, the same search runs on Apple's iTunes Search API,
which also has 30-second previews. Checked on Sam Cooke's version, which both have:
the features from the two previews differ only slightly (e.g. acousticness 0.74 vs
0.85, tempo 118 vs 115), far less than the songs differ from each other.

In [3]:
matches, newly_matched = update_matches(songs)   # only looks up songs not matched before
print(f"{len(newly_matched)} newly matched")

report = match_report(songs, matches)            # most doubtful matches first
unmatched = report[report["source"].isna()]
if len(unmatched):
    print("No match on Deezer or iTunes for:", "; ".join(unmatched["artist"] + " - " + unmatched["title"]))
report.drop(columns=["track_id"])

0 newly matched


,artist,title,source,source_id,match_title,album,length_diff_s
0,Nina Simone,Summertime,deezer,1137981742,Nina Simone - Summertime,Soft Silk - My Baby Just Cares For Me,23.0
1,Albert Ayler,Summertime,deezer,1502698702,Albert Ayler - Summertime,Spiritual Jazz 4: Americans in Europe,13.0
2,The Ventures,Summertime,deezer,1526609722,The Ventures - Summertime,Beach Party,-6.0
3,Sidney Bechet,Summertime,deezer,4202660872,Sidney Bechet - Summertime (2026 Remaster),Classics in Swingtime (2026 Remaster),5.0
4,Lainie Kazan,Summertime,deezer,2595493372,Lainie Kazan - Summertime,Lainie Kazan,-4.0
5,Big Brother & The Holding Company;Janis Joplin,Summertime,deezer,15613154,Big Brother & The Holding Company - Summertime,The Essential Janis Joplin,-3.0
6,Lena Horne,Summertime,deezer,964201,"Lena Horne - Summertime (From ""Porgy and Bess"")",Stormy Weather (Expanded Edition),-3.0
7,Mahalia Jackson,Summertime / Sometimes I Feel Like a Motherles...,deezer,106249326,Mahalia Jackson - Summertime / Sometimes I Fee...,Bless This House,-3.0
8,Miles Davis,Summertime - Mono Version,deezer,872356,Miles Davis - Summertime,Porgy And Bess,-2.0
9,The Buckinghams,Summertime,deezer,142748114,The Buckinghams - Summertime,Kind of a Drag (Expanded Edition),2.0


## 3. Audio features from each preview

Code: `summertime/features.py`.

ReccoBeats' full-track features covered only 6 of the first 11 recordings, and features
computed from a 30-second clip don't agree with full-track ones (tempo and
acousticness especially), so mixing the two would distort the map. Instead, **every**
song is analyzed the same way: its preview is uploaded to ReccoBeats'
audio-analysis endpoint.

So the map describes the 30-second excerpt, which is also what the website plays.
The catch: if a preview happens to be an intro or a solo, it may not represent the
whole performance.

Results are cached in `data/features.csv`; delete a row to re-analyze that song.

In [4]:
_, newly_analyzed = update_features(matches)     # only analyzes previews not analyzed before
print(f"{newly_analyzed} newly analyzed")

df = load_dataset()                              # songs + match + features, one row per song
print(f"{len(df)} recordings with features")
df[["artist", "title", "year", *FEATURES]].round(2)

0 newly analyzed
54 recordings with features


,artist,title,year,danceability,energy,valence,tempo,loudness,acousticness,instrumentalness,speechiness,liveness
0,Big Brother & The Holding Company;Janis Joplin,Summertime,1968,0.51,0.40,0.45,101.33,-9.89,0.26,0.01,0.04,0.14
1,Ella Fitzgerald;Louis Armstrong,Summertime,1957,0.30,0.09,0.14,111.75,-15.89,0.97,0.88,0.04,0.10
2,The Zombies,Summertime,1965,0.49,0.20,0.20,107.24,-15.16,0.84,0.02,0.04,0.11
3,Sam Cooke,Summertime,1957,0.60,0.33,0.45,115.09,-11.82,0.85,0.04,0.04,0.14
4,Sublime,Doin' Time,1996,0.63,0.80,0.67,71.68,-3.66,0.03,0.00,0.11,0.17
5,Charlie Parker,Summertime,1949,0.43,0.23,0.48,140.65,-10.35,0.97,0.06,0.04,0.13
6,Al Green,Summertime,1969,0.62,0.29,0.53,137.08,-12.75,0.79,0.00,0.05,0.13
7,Lana Del Rey,Doin' Time,2019,0.65,0.53,0.48,145.87,-9.27,0.48,0.01,0.04,0.13
8,Doc Watson;David Grisman,Summertime,1997,0.57,0.16,0.39,131.02,-14.57,0.94,0.02,0.04,0.12
9,Billy Stewart,Summertime,1966,0.71,0.66,0.81,111.59,-8.10,0.42,0.00,0.06,0.14


## 4. Standardize

Features are on very different scales (tempo in BPM, loudness in dB, the rest 0–1). Z-score each one so no feature dominates just because of its units.

In [5]:
# TODO

## 5. PCA: the 2-D map

Project the nine standardized features down to two axes, plot each recording as a labelled point, and draw arrows showing which features drive each axis.

In [6]:
# TODO

## 6. K-Means: are there natural groups?

Cluster on all nine standardized features (not on the 2-D map), trying a few values of k and comparing silhouette scores.

In [7]:
# TODO

## 7. How valid are the groups?

Silhouette score per cluster, plus a stability test: re-cluster random subsets of the songs and measure how often the same recordings end up together.

In [8]:
# TODO

## 8. What might each group represent?

Compare each cluster's average features with the whole set and draft a plain-language label from the biggest differences. Edit these by ear.

In [9]:
# TODO

## 9. Export for the website

Write songs, coordinates, clusters, album covers and preview links to `site/data.json`.

In [10]:
# TODO